# LegalQA v9.2 — evidence-grounded QLoRA trên official train
`TRAIN_SCOPE='split'` giữ 400 câu grouped-dev để đo candidate oracle trung thực. Sau khi chốt cấu hình, dùng `TRAIN_SCOPE='full'` để tạo adapter public cuối cùng. Checkpoint chứa model adapter, optimizer, scheduler và RNG; có thể attach output cũ để resume.

In [ ]:
TRAIN_SCOPE = 'split'       # 'split' để calibration; 'full' cho adapter public cuối
DEV_SIZE = 400
EPOCHS = 1.0
GRAD_ACCUM = 16
MAX_LENGTH = 1536
MAX_TARGET_TOKENS = 700
MAX_EVIDENCE_CHARS = 5000
LORA_R = 8
CHECKPOINT_STEPS = 20
SAVE_TOTAL_LIMIT = 2
MAX_TRAIN_SAMPLES = None    # đặt 300 để smoke; None là toàn bộ split/full
AUTO_RESUME_TRAINING = True


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
input_root = Path('/kaggle/input')
EXPECTED_SOURCE_CONTRACT = {
    'schema': 'legalqa-source-contract-v1',
    'release': '0.9.2',
    'agentic_namespace': 'agentic-v3',
    'cache_schema': 'inference-v7-agentic-error-isolation',
    'selector_schema': 'selector-weights-v3-agentic-evidence',
    'offline_wheel_count': 12,
}
REQUIRED_WHEELS = ('transformers-', 'tokenizers-', 'huggingface_hub-', 'hf_xet-', 'safetensors-', 'accelerate-', 'peft-', 'bitsandbytes-', 'sentence_transformers-', 'sentencepiece-', 'regex-', 'click-')
def source_contract_errors(path):
    errors = []
    marker = path / 'SOURCE_CONTRACT.json'
    try:
        contract = json.loads(marker.read_text(encoding='utf-8'))
    except Exception as exc:
        return [f'thiếu/hỏng SOURCE_CONTRACT.json: {exc}']
    for key, expected in EXPECTED_SOURCE_CONTRACT.items():
        if contract.get(key) != expected:
            errors.append(f'{key}={contract.get(key)!r}, cần {expected!r}')
    required_files = (path / 'kaggle' / 'run_qwen_qlora.py', path / 'kaggle' / 'run_inference_only.py', path / 'src' / 'legalqa' / 'selector_schema.py', path / 'RELEASE_V9_2.md')
    errors.extend(f'thiếu {file.relative_to(path)}' for file in required_files if not file.is_file())
    wheels = list((path / 'wheelhouse').glob('*.whl'))
    names = [wheel.name for wheel in wheels]
    missing_wheels = [prefix for prefix in REQUIRED_WHEELS if not any(name.startswith(prefix) for name in names)]
    if len(wheels) != 12 or missing_wheels or any(wheel.stat().st_size < 10_000 for wheel in wheels):
        errors.append(f'wheelhouse không hợp lệ: count={len(wheels)}, missing={missing_wheels}')
    return errors
source_projects = sorted({p.parent for p in input_root.rglob('pyproject.toml')})
source_reports = {str(path): source_contract_errors(path) for path in source_projects}
qualified_sources = [path for path in source_projects if not source_reports[str(path)]]
if len(qualified_sources) != 1:
    raise RuntimeError(f'Cần đúng một source v9.2 reviewed; source cũ nhúng trong QLoRA sẽ bị loại. qualified={qualified_sources}; reports={source_reports}')
selected_source = qualified_sources[0]
print('Source v9.2 reviewed:', selected_source)
print('Rejected source copies:', {path: reasons for path, reasons in source_reports.items() if reasons})
project = Path('/kaggle/working/legalqa_source')
if project.exists():
    shutil.rmtree(project)
shutil.copytree(selected_source, project)
wheels = sorted(str(path) for path in (project / 'wheelhouse').glob('*.whl'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--upgrade', *wheels], check=True)
os.environ['PYTHONPATH'] = str(project / 'src')
os.environ['PYTHONUTF8'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], check=True)
print('Source:', selected_source)


In [ ]:
resume_roots = []
if AUTO_RESUME_TRAINING:
    for signature in input_root.rglob('resume_signature.json'):
        scope_file = signature.parent / 'qlora_scope.json'
        if scope_file.is_file() and __import__('json').loads(scope_file.read_text(encoding='utf-8')).get('train_scope') == TRAIN_SCOPE and any(p.is_dir() and p.name.startswith('checkpoint-') for p in signature.parent.iterdir()):
            resume_roots.append(signature.parent)
resume_roots = sorted(set(resume_roots))
if len(resume_roots) > 1:
    raise RuntimeError(f'Chỉ attach một QLoRA state phù hợp: {resume_roots}')
command = [
    sys.executable, str(project / 'kaggle' / 'run_qwen_qlora.py'),
    '--config', str(project / 'configs' / 'kaggle_inference_t4.yaml'),
    '--input-root', '/kaggle/input',
    '--working', '/kaggle/working/legalqa_qlora',
    '--train-scope', TRAIN_SCOPE,
    '--dev-size', str(DEV_SIZE),
    '--epochs', str(EPOCHS),
    '--grad-accum', str(GRAD_ACCUM),
    '--max-length', str(MAX_LENGTH),
    '--max-target-tokens', str(MAX_TARGET_TOKENS),
    '--max-evidence-chars', str(MAX_EVIDENCE_CHARS),
    '--lora-r', str(LORA_R),
    '--checkpoint-steps', str(CHECKPOINT_STEPS),
    '--save-total-limit', str(SAVE_TOTAL_LIMIT),
]
if MAX_TRAIN_SAMPLES is not None:
    command.extend(['--max-train-samples', str(MAX_TRAIN_SAMPLES)])
if resume_roots:
    command.extend(['--resume-input', str(resume_roots[0])])
print(' '.join(command))
subprocess.run(command, check=True)


In [ ]:
import json
root = Path('/kaggle/working/legalqa_qlora')
manifest = root / 'qlora_manifest.json'
assert manifest.exists(), 'Training chưa hoàn tất. Hãy Save Version để giữ checkpoint rồi resume.'
payload = json.loads(manifest.read_text(encoding='utf-8'))
adapter = Path(payload['adapter'])
assert (adapter / 'adapter_config.json').is_file()
assert (adapter / 'training_complete.json').is_file()
assert (root / 'qwen_qlora_adapter.zip').is_file()
print(json.dumps(payload, ensure_ascii=False, indent=2))
